# NOUS-CLS · The Librarian on **code models** (Colab / GPU)

Continual learning over a stream of **programming languages** (Python → Java → Go → …),
using frozen **code-model embeddings** as concept addresses. The *librarian* is a
memory policy over prototypes — surprise-spawn + evidence-based consolidation +
frozen semantic id + a distance/entropy **defer** gate — compared to a `naive`
one-prototype-per-observation baseline.

**The diff we test:** run the *identical* continual-learning task with an **older**
code model vs a **newer** one, and see whether the newer model gives cleaner
routing + retention. Decoder code-LLMs need **whitening** (their raw embeddings are
anisotropic); encoders don't.

Runtime → set **Runtime ▸ Change runtime type ▸ GPU**.

In [ ]:
%pip -q install "transformers>=4.44" "datasets>=2.19" torch --upgrade

In [ ]:
import torch, torch.nn.functional as F, math, time
from transformers import AutoModel, AutoTokenizer
DEV = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", DEV)

In [ ]:
# ── The librarian: prototype memory over embeddings ──────────────────────────
def entropy(centers, e, temp=0.3):
    d2 = ((centers - e) ** 2).sum(-1)
    p = torch.softmax(-d2 / temp, dim=-1)
    return float(-(p * (p + 1e-12).log()).sum())

class FeatureLibrarian:
    """full=True: surprise-spawn + evidence consolidation (k hits, >=c agreement)
    -> frozen id, + defer on near+ambiguous. full=False: one prototype per obs."""
    def __init__(self, full=True, k=3, c=0.5, radius=0.7, tau=0.5, temp=0.3):
        self.full,self.k,self.c,self.radius,self.tau,self.temp = full,k,c,radius,tau,temp
        self.ids, self.prov, self.n_defer = [], [], 0
    def _nearest(self, items, e):
        if not items: return None, math.inf
        d = [((it["center"]-e)**2).sum().item() for it in items]
        j = min(range(len(d)), key=lambda i: d[i]); return j, d[j]**0.5
    def is_ambiguous(self, e):
        if len(self.ids) < 2: return False
        C = torch.stack([it["center"] for it in self.ids]); _, dist = self._nearest(self.ids, e)
        return dist < self.radius and entropy(C, e, self.temp) > self.tau
    def observe(self, e, y):
        if self.full and self.is_ambiguous(e): self.n_defer += 1; return
        if not self.full: self.ids.append({"center": e.clone(), "label": y}); return
        j, dist = self._nearest(self.ids, e)
        if j is not None and dist < self.radius: return          # near a known id -> ignore labile obs
        pj, _ = self._nearest(self.prov, e)
        if pj is not None and ((self.prov[pj]["center"]-e)**2).sum().sqrt().item() < self.radius:
            p = self.prov[pj]; p["hits"] += 1; p["counts"][y] = p["counts"].get(y,0)+1
            p["center"] = p["center"] + 0.5*(e - p["center"])
        else:
            self.prov.append({"center": e.clone(), "counts": {y:1}, "hits":1}); pj = len(self.prov)-1
        p = self.prov[pj]
        if p["hits"] >= self.k and max(p["counts"].values())/p["hits"] >= self.c:
            self.ids.append({"center": p["center"], "label": max(p["counts"], key=p["counts"].get)})
            self.prov.pop(pj)
    def predict(self, e):
        j, _ = self._nearest(self.ids, e); return self.ids[j]["label"] if j is not None else -1

In [ ]:
# ── Code embeddings (mean-pool, whitened with TRAIN stats) ──────────────────
@torch.no_grad()
def embed(model, tok, texts, maxlen=128, bs=32):
    out = []
    for i in range(0, len(texts), bs):
        e = tok(texts[i:i+bs], return_tensors="pt", padding=True, truncation=True, max_length=maxlen).to(DEV)
        h = model(**e).last_hidden_state
        mm = e["attention_mask"].unsqueeze(-1).float()
        out.append(F.normalize((h*mm).sum(1)/mm.sum(1), dim=-1).cpu())
    return torch.cat(out)

def features(model_name, train_texts, test_texts):
    tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    m = AutoModel.from_pretrained(model_name, trust_remote_code=True).to(DEV).eval()
    Xtr, Xte = embed(m, tok, train_texts), embed(m, tok, test_texts)
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-6                       # whiten (fixes decoder-LM anisotropy)
    W = lambda X: F.normalize((X - mu)/sd, dim=-1)
    del m; torch.cuda.empty_cache() if DEV=="cuda" else None
    return W(Xtr), W(Xte)

In [ ]:
# ── Code stream: CodeSearchNet by language ──────────────────────────────────
from datasets import load_dataset
LANGS = ["python", "java", "javascript", "go", "php", "ruby"]
PER, TEST = 150, 60
def load_code(seed=0):
    g = torch.Generator().manual_seed(seed); tr, te, ytr, yte = [], [], [], []
    for li, lang in enumerate(LANGS):
        ds = load_dataset("code_search_net", lang, split="train", trust_remote_code=True)
        col = "func_code_string" if "func_code_string" in ds.column_names else "whole_func_string"
        idx = torch.randperm(len(ds), generator=g)[:PER+TEST].tolist()
        docs = [ds[i][col] for i in idx]
        tr += docs[:PER]; ytr += [li]*PER; te += docs[PER:]; yte += [li]*TEST
    return tr, te, torch.tensor(ytr), torch.tensor(yte)
train_texts, test_texts, ytr, yte = load_code()
print("languages:", LANGS, "| train", len(train_texts), "test", len(test_texts))

In [ ]:
# ── Run the librarian continual stream for one model ────────────────────────
def ncm_routing(Xtr, ytr, Xte, yte):
    mu = torch.stack([Xtr[ytr==c].mean(0) for c in range(len(LANGS))])
    return (torch.cdist(Xte, mu).argmin(1) == yte).float().mean().item()

def run_model(model_name, seed=0, epochs=6, noise=0.15):
    Xtr, Xte = features(model_name, train_texts, test_texts)
    routing = ncm_routing(Xtr, ytr, Xte, yte)
    g = torch.Generator().manual_seed(seed+3)
    res = {}
    for kind in ("librarian", "naive"):
        lib = FeatureLibrarian(full=(kind=="librarian"))
        by = [Xtr[ytr==c] for c in range(len(LANGS))]
        for ep in range(epochs):
            active = range(len(LANGS)) if ep >= epochs//2 else range(len(LANGS)-1)  # last lang novel
            order = [(e, c) for c in active for e in by[c]]
            for k in torch.randperm(len(order), generator=g):
                e, y = order[k]
                if torch.rand(1, generator=g).item() < noise:
                    y = int(torch.randint(0, len(LANGS), (1,), generator=g).item())
                lib.observe(e, y)
        clean = (torch.tensor([lib.predict(e) for e in Xte]) == yte).float().mean().item()
        nmask = yte == (len(LANGS)-1)
        novel = (torch.tensor([lib.predict(Xte[i]) for i in range(len(Xte)) if nmask[i]]) == yte[nmask]).float().mean().item()
        amb = 0.0
        if kind=="librarian" and len(lib.ids) >= 2:
            C = torch.stack([it["center"] for it in lib.ids])
            probes = [(lib.ids[i]["center"]+lib.ids[j]["center"])/2
                      for i,j in [torch.randint(0,len(lib.ids),(2,),generator=g).tolist() for _ in range(60)] if i!=j]
            amb = sum(entropy(C,p,lib.temp) > lib.tau for p in probes)/max(len(probes),1)
        res[kind] = dict(clean=clean, novel=novel, ids=len(lib.ids), defer=amb)
    return dict(routing=routing, **res)

In [ ]:
# ── The diff: older code model vs newer code LLM ────────────────────────────
MODELS = {
    "codebert-base (2020, 125M encoder)": "microsoft/codebert-base",
    "Qwen2.5-Coder-1.5B (2024, code LLM)": "Qwen/Qwen2.5-Coder-1.5B",
    # add/swap freely, e.g. "jinaai/jina-embeddings-v2-base-code"
}
rows = []
for label, name in MODELS.items():
    t=time.time(); r = run_model(name); print(f"[{time.time()-t:.0f}s] {label}")
    L, N = r["librarian"], r["naive"]
    rows.append((label, r["routing"], L["clean"], L["novel"], L["ids"], L["defer"], N["clean"], N["ids"]))

print("\n%-38s %7s | %-26s | %-18s" % ("model", "routing", "LIBRARIAN clean/novel/ids/defer", "naive clean/ids"))
for label, rt, lc, ln, li, ld, nc, ni in rows:
    print("%-38s %6.2f  | %.2f / %.2f / %4d / %.2f      | %.2f / %5d" % (label, rt, lc, ln, li, ld, nc, ni))
print("\nDiff = does the newer code model raise routing + librarian retention?")